In [1]:
%reset -f

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

Check for cuda

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

Using device: cuda


Load data

In [4]:
path = Path.cwd().parent.parent / "TabulatedData" / "values_all_layers_16bit.parquet"

dataset = pd.read_parquet(path)

X = dataset.drop("name", axis=1).values
y = dataset["name"].values

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
y_one_hot = encoder.fit_transform(pd.DataFrame(y))
y = np.argmax(y_one_hot, axis=1)

# Get class names from encoder
class_names = encoder.categories_[0]

print("Data Loaded")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42
)

del dataset, X, y

# -----------------------------
# Feature Scaling (recommended for MLP)
# -----------------------------
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

Data Loaded


To PyTorch Tensors

In [6]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


X_train = X_train.to(device)
X_test = X_test.to(device)
y_train = y_train.to(device)
y_test = y_test.to(device)

train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)

/tmp/ipykernel_31149/1074341973.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_train = torch.tensor(X_train, dtype=torch.float32)
/tmp/ipykernel_31149/1074341973.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_test = torch.tensor(X_test, dtype=torch.float32)
/tmp/ipykernel_31149/1074341973.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y_train = torch.tensor(y_train, dtype=torch.long)
/tmp/ipykernel_31149/1074341973.py:5: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone()

Define Model

In [7]:
input_size = X_train.shape[1]
num_classes = len(class_names)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_size, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, 50),
            nn.ReLU(),
            nn.Linear(50, num_classes)
        )

    def forward(self, x):
        return self.net(x)

model = MLP().to(device)

# -----------------------------
# Loss and Optimizer
# -----------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

Training Loop

In [8]:
epochs = 1000

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss:.4f}")


Epoch [100/1000], Loss: 3.4938
Epoch [200/1000], Loss: 9.4372
Epoch [300/1000], Loss: 11.5521
Epoch [400/1000], Loss: 11.1045
Epoch [500/1000], Loss: 13.6516
Epoch [600/1000], Loss: 7.0291
Epoch [700/1000], Loss: 7.8259
Epoch [800/1000], Loss: 9.5738
Epoch [900/1000], Loss: 14.9997
Epoch [1000/1000], Loss: 11.1736


Evaluation Test

In [9]:
model.eval()

with torch.no_grad():
    outputs = model(X_test)
    preds = torch.argmax(outputs, dim=1)

acc = accuracy_score(y_test.cpu().numpy(), preds.cpu().numpy())

print(f"\nTest Accuracy: {acc:.4f}")
print(classification_report(y_test.cpu().numpy(), preds.cpu().numpy(), target_names=class_names, zero_division=0))


Test Accuracy: 0.2576
                    precision    recall  f1-score   support

      Aditya Kundu       0.70      0.35      0.47        20
       Akash Gupta       0.37      0.34      0.35        59
          Ankur De       0.57      0.62      0.59       187
           Ashtavi       0.42      0.47      0.44        32
          Avyuktha       0.09      0.11      0.10       105
        Harshith H       0.32      0.33      0.33        30
     Jiya Sachdeva       0.36      0.34      0.35        94
     Karthikeya SK       0.13      0.20      0.16        20
    Nandini Sharma       0.57      0.33      0.42        49
           Navnita       0.06      0.06      0.06       104
          Papia De       0.67      0.51      0.58        47
        Pavithra S       0.54      0.58      0.56        95
            Piyali       0.62      0.36      0.46        22
Prajwal Mundiganal       0.79      0.73      0.76        52
           Ruthvik       0.42      0.57      0.48        96
            Tris

Evaluation Train

In [11]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)
    preds = torch.argmax(outputs, dim=1)

acc = accuracy_score(y_train.cpu().numpy(), preds.cpu().numpy())

print(f"\nTest Accuracy: {acc:.4f}")


Test Accuracy: 0.9790
